In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
from peft import LoraConfig, get_peft_model, PeftModel
import torch


device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model_id = "yoakiyama/MSA-Pairformer"


from MSA_Pairformer.model import MSAPairformer
from MSA_Pairformer.dataset import MSA

msa_pairformer = MSAPairformer.from_pretrained(device=device)



/home/mooolab/anaconda3/envs/yoosun/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 123361.88it/s]


In [2]:
print(msa_pairformer.device)
print(sum(p.numel() for p in msa_pairformer.parameters()) / 1e6, "M parameters")


cuda:1
111.365468 M parameters


In [3]:
# [진단용 셀]
# LoRA 적용 전에 이 셀을 먼저 실행해 보세요.

# 모델의 모듈 중 하나를 직접 출력해봅니다.
print(msa_pairformer.core_stack.final_msa_pwa.to_out[1])

LinearNoBias(in_features=256, out_features=464, bias=False)


In [4]:
for name, module in msa_pairformer.named_modules():
    if "to_out" in name:
        print(name, type(module))


core_stack.layers.0.0.to_out <class 'torch.nn.modules.container.Sequential'>
core_stack.layers.0.0.to_out.0 <class 'einops.layers.torch.Rearrange'>
core_stack.layers.0.0.to_out.1 <class 'MSA_Pairformer.core.LinearNoBias'>
core_stack.layers.0.0.to_out.2 <class 'MSA_Pairformer.core.Dropout'>
core_stack.layers.0.0.to_out.2.dropout <class 'torch.nn.modules.dropout.Dropout'>
core_stack.layers.0.3.tri_mult_outgoing.fn.to_out_norm <class 'torch.nn.modules.normalization.LayerNorm'>
core_stack.layers.0.3.tri_mult_outgoing.fn.to_out <class 'torch.nn.modules.container.Sequential'>
core_stack.layers.0.3.tri_mult_outgoing.fn.to_out.0 <class 'MSA_Pairformer.core.LinearNoBias'>
core_stack.layers.0.3.tri_mult_outgoing.fn.to_out.1 <class 'MSA_Pairformer.core.Dropout'>
core_stack.layers.0.3.tri_mult_outgoing.fn.to_out.1.dropout <class 'torch.nn.modules.dropout.Dropout'>
core_stack.layers.0.3.tri_mult_incoming.fn.to_out_norm <class 'torch.nn.modules.normalization.LayerNorm'>
core_stack.layers.0.3.tri_mul

In [5]:
import torch.nn as nn
from peft import LoraConfig, get_peft_model

# 1️⃣ 모든 모듈 이름 출력해서 dropout 있는 애 걸러내기
dropout_names = [name for name, m in msa_pairformer.named_modules() if isinstance(m, nn.Dropout)]
dropout_names_set = set(dropout_names)

# 2️⃣ LoRA 타깃 후보
base_target_modules = [
    "msa_to_values_and_gates.1",
    "pairwise_repr_to_attn.1",
    "to_out.1",
    "lm_head.dense",
]

# 3️⃣ Dropout 이름이 겹치는 경우 자동 제외
safe_targets = []
for t in base_target_modules:
    if not any(t in d for d in dropout_names_set):
        safe_targets.append(t)

print(f" 최종 LoRA 타깃 모듈: {safe_targets}")

# 4️⃣ LoRA 설정 및 적용
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=safe_targets,
    lora_dropout=0.1,
    bias="none",
)

lora_model = get_peft_model(msa_pairformer, lora_config)
lora_model.print_trainable_parameters()


 최종 LoRA 타깃 모듈: ['msa_to_values_and_gates.1', 'pairwise_repr_to_attn.1', 'lm_head.dense']
trainable params: 239,296 || all params: 111,604,764 || trainable%: 0.2144


In [13]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from MSA_Pairformer.dataset import MSA, msa_mlm, aa2tok_d
from torch.utils.checkpoint import checkpoint

# 🔹 1. 디바이스 설정
device = next(lora_model.parameters()).device

# 🔹 2. 모델을 bf16으로 변환
lora_model = lora_model.to(torch.bfloat16)
lora_model.train()

# 🔹 3. MSA 데이터 불러오기
msa_path = "X5L4P3.a3m"
msa_data = MSA(msa_path, diverse_select_method="none")
msa_t = msa_data.diverse_tokenized_msa.unsqueeze(0).to(device)

# 🔹 4. 마스킹 (mutate_pssm=False)
masked_msa, mlm_indices = msa_mlm(
    msa_t.cpu(),
    mask_prob=0.15,
    mutate_pssm=False
)
masked_msa = masked_msa.to(device)

# 🔹 5. one-hot 인코딩 (bf16)
msa_onehot = torch.nn.functional.one_hot(masked_msa, num_classes=28).to(torch.bfloat16).to(device)

# 🔹 6. forward (autocast로 bf16 연산)
with torch.cuda.amp.autocast(dtype=torch.bfloat16):
    output = lora_model(msa=msa_onehot)
    logits = output["logits"]  # (B, 1, L, vocab)

    # query-only masking
    mask = (masked_msa[:, 0, :] == aa2tok_d["<mask>"])  # (B, L)
    labels = msa_t[:, 0, :]  # (B, L)

    masked_logits = logits[:, 0, :, :][mask]
    masked_labels = labels[mask]

    # 🔹 7. CrossEntropy는 fp32로 계산 (정확도 유지용)
    loss = F.cross_entropy(masked_logits.float(), masked_labels.long())

print(f"✅ Loss computed successfully (bf16): {loss.item():.4f} | masked tokens: {mask.sum().item()}")


/tmp/ipykernel_1894343/1382909973.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 1 has a total capacity of 23.52 GiB of which 3.31 MiB is free. Process 1355230 has 3.30 GiB memory in use. Process 1507598 has 5.27 GiB memory in use. Including non-PyTorch memory, this process has 14.92 GiB memory in use. Of the allocated memory 13.54 GiB is allocated by PyTorch, and 940.59 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from MSA_Pairformer.dataset import MSA, msa_mlm, aa2tok_d


device = next(lora_model.parameters()).device

msa_path = "X5L4P3.a3m"
msa_data = MSA(msa_path, diverse_select_method="none")
msa_t = msa_data.diverse_tokenized_msa.unsqueeze(0).to(device)

# 마스킹 (mutate_pssm=False)
masked_msa, mlm_indices = msa_mlm(
    msa_t.cpu(),
    mask_prob=0.15,
    mutate_pssm=False
)
masked_msa = masked_msa.to(device)

# one-hot 인코딩
msa_onehot = torch.nn.functional.one_hot(masked_msa, num_classes=28).float().to(device)

# forward
output = lora_model(msa=msa_onehot)
logits = output["logits"]  # (B, 1, L, vocab)

# query-only masking
mask = (masked_msa[:, 0, :] == aa2tok_d["<mask>"])  # (B, L)
labels = msa_t[:, 0, :]  # (B, L)
masked_logits = logits[:, 0, :, :][mask]
masked_labels = labels[mask]

loss = F.cross_entropy(masked_logits, masked_labels)
print(f"✅ Loss computed successfully: {loss.item():.4f} | masked tokens: {mask.sum().item()}")


RuntimeError: expected mat1 and mat2 to have the same dtype, but got: float != c10::BFloat16

In [9]:
from MSA_Pairformer.dataset import MSA

msa_path = "X5L4P3.a3m"  # 실제 파일 경로
msa_data = MSA(msa_path)


#토큰화된 MSA 가져오기
msa_tensor = msa_data.diverse_tokenized_msa  # (depth x length)
print("MSA tensor shape:", msa_tensor.shape)

#batch 차원 추가 + GPU로 이동
msa_tensor = msa_tensor.unsqueeze(0).to(lora_model.device)  # (1, depth, length)

#one-hot 인코딩
msa_onehot = torch.nn.functional.one_hot(msa_tensor, num_classes=28).float()
print("One-hot shape:", msa_onehot.shape)  # (1, depth, length, 28)

#모델 forward
with torch.no_grad():
    output = lora_model(
        msa=msa_onehot,
        return_contacts=False,
        query_only=True
    )

print("forward ok, logits shape:", output["logits"].shape)

MSA tensor shape: torch.Size([1, 120])
One-hot shape: torch.Size([1, 1, 120, 28])


OutOfMemoryError: CUDA out of memory. Tried to allocate 30.00 MiB. GPU 0 has a total capacity of 23.52 GiB of which 17.50 MiB is free. Process 2595 has 392.70 MiB memory in use. Process 1870514 has 8.76 GiB memory in use. Including non-PyTorch memory, this process has 14.32 GiB memory in use. Of the allocated memory 13.26 GiB is allocated by PyTorch, and 627.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
# from scipy.stats import spearmanr
# import torch.nn.functional as F
# import pandas as pd

# # 1️⃣ load DMS data
# dms_df = pd.read_csv("ProteinGym_X5L4P3.csv")

# # 2️⃣ baseline wild-type log likelihood
# msa_wt = msa_data.diverse_tokenized_msa.unsqueeze(0).to(lora_model.device)
# onehot_wt = torch.nn.functional.one_hot(msa_wt, num_classes=28).float()
# logits_wt = lora_model(msa=onehot_wt)["logits"]
# log_probs_wt = F.log_softmax(logits_wt, dim=-1)

# # 3️⃣ mutant score
# pred_scores = []
# for _, row in dms_df.iterrows():
#     pos = int(row["Position"]) - 1
#     wt_tok = aa2tok_d[row["WT"]]
#     mut_tok = aa2tok_d[row["Mut"]]

#     msa_mut = msa_wt.clone()
#     msa_mut[0, 0, pos] = mut_tok  # mutate query sequence
#     onehot_mut = torch.nn.functional.one_hot(msa_mut, num_classes=28).float()
#     logits_mut = lora_model(msa=onehot_mut)["logits"]
#     log_probs_mut = F.log_softmax(logits_mut, dim=-1)

#     # difference in log-likelihood at the mutated position
#     delta_logP = (log_probs_mut[0, 0, pos, mut_tok] - log_probs_wt[0, 0, pos, wt_tok]).item()
#     pred_scores.append(delta_logP)

# # 4️⃣ correlation
# rho, pval = spearmanr(pred_scores, dms_df["fitness"])
# print(f"Spearman's ρ: {rho:.3f} (p={pval:.2e})")
